# Archetype clustering experimentation: PCA + Gaussian Mixture

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.ml.archetype_clustering import (
    ArchetypeClusteringConfig,
    ArchetypeClusteringConfigsByRole,
    fit_archetype_clustering,
    numeric_feature_columns,
    prepare_dataframe_for_archetype_clustering,
)
from src.ml.archetype_finetune import (
    grid_sweep_pca_and_gmm,
    pca_cumulative_variance,
    scaled_feature_matrix,
)
from src.pipeline.s3_interaction import gold_player_year_output_key, read_parquet_from_s3

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

## Pitchers — data loading

In [ ]:
ROLE = "pitcher"
YEAR = 2025

S3_BUCKET = "diamond-dna"
GOLD_PREFIX = "gold/statcast"

LOCAL_GOLD_PARQUET: Path | None = None 

RANDOM_STATE = 42
N_INIT = 10
GMM_COVARIANCE_TYPE = "full"

In [ ]:
def load_gold_player_year(
    *,
    role: str,
    year: int,
    bucket: str,
    gold_prefix: str,
    local_parquet: Path | None,
) -> pd.DataFrame:
    if local_parquet is not None and local_parquet.is_file():
        df = pd.read_parquet(local_parquet)
    else:
        key = gold_player_year_output_key(gold_prefix, role, year)
        df = read_parquet_from_s3(bucket, key, missing_key_log="warning")
        if df is None:
            raise FileNotFoundError(f"No gold parquet at s3://{bucket}/{key}")
    if "role" not in df.columns:
        df = df.copy()
        df["role"] = role
    return df


gold_df = load_gold_player_year(
    role=ROLE,
    year=YEAR,
    bucket=S3_BUCKET,
    gold_prefix=GOLD_PREFIX,
    local_parquet=LOCAL_GOLD_PARQUET,
)
gold_df.shape

### Pitchers — feature columns (production rules)

In [ ]:
df_idx = prepare_dataframe_for_archetype_clustering(gold_df)
feat_cols = numeric_feature_columns(df_idx)
print(f"n_features = {len(feat_cols)}")
feat_cols

### Pitchers — cumulative PCA variance

Same probe as the KMeans notebook.

In [ ]:
MAX_PC_PROBE = 50

X_scaled, _cols_used, _scaler = scaled_feature_matrix(gold_df)
assert _cols_used == feat_cols

cum, ratios = pca_cumulative_variance(
    X_scaled, max_components=MAX_PC_PROBE, random_state=RANDOM_STATE
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(cum) + 1), cum, "o-")
ax.axhline(0.80, color="C1", ls="--", label="80%")
ax.axhline(0.85, color="C2", ls="--", label="85%")
ax.set_xlabel("n_components")
ax.set_ylabel("cumulative explained variance")
ax.legend()
ax.set_title("PITCHERS: PCA variance vs n_components")
plt.tight_layout()
plt.show()

### Pitchers — grid sweep: PCA size × GMM `n_components`

In [ ]:
PCA_N_COMPONENTS_CANDIDATES_PITCHER = [12, 14, 16, 18, 20]
K_MIN_PITCHER = 6
K_MAX_PITCHER = 12
grid_gmm_pitcher = grid_sweep_pca_and_gmm(
    X_scaled,
    pca_n_components_list=PCA_N_COMPONENTS_CANDIDATES_PITCHER,
    k_min=K_MIN_PITCHER,
    k_max=K_MAX_PITCHER,
    random_state=RANDOM_STATE,
    n_init=N_INIT,
    covariance_type=GMM_COVARIANCE_TYPE,
)

grid_gmm_pitcher.sort_values("gmm_bic", ascending=True).head(25)

In [ ]:
pick = grid_gmm_pitcher.sort_values("gmm_bic", ascending=True).iloc[0]
PCA_N = int(pick["pca_n_components"])
K = int(pick["k"])
print("Suggested (min BIC in grid): PCA_N =", PCA_N, ", n_components =", K)

prod_cfg = ArchetypeClusteringConfig(
    pca_n_components=PCA_N,
    n_clusters=K,
    random_state=RANDOM_STATE,
    n_init=N_INIT,
    covariance_type=GMM_COVARIANCE_TYPE,
)
prod_cfg

### Pitchers — local fit (matches pipeline)

In [ ]:
labeled, meta, bundle = fit_archetype_clustering(
    gold_df, role=ROLE, year=YEAR, config=prod_cfg
)
print("keys:", meta["clustering_method"], "BIC", round(meta["gmm_bic"], 2))
labeled[["player_id", "year", "role", "cluster_id"]].head()

## Batters — same workflow as pitchers

In [ ]:
ROLE_BAT = "batter"
gold_df_bat = load_gold_player_year(
    role=ROLE_BAT,
    year=YEAR,
    bucket=S3_BUCKET,
    gold_prefix=GOLD_PREFIX,
    local_parquet=LOCAL_GOLD_PARQUET,
)
print("batter shape:", gold_df_bat.shape)

In [ ]:
df_idx_bat = prepare_dataframe_for_archetype_clustering(gold_df_bat)
feat_cols_bat = numeric_feature_columns(df_idx_bat)
print(f"[batter] n_features = {len(feat_cols_bat)}")


In [ ]:
X_scaled_bat, cols_b, _ = scaled_feature_matrix(gold_df_bat)
assert cols_b == feat_cols_bat

cum_b, _ = pca_cumulative_variance(
    X_scaled_bat, max_components=MAX_PC_PROBE, random_state=RANDOM_STATE
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(cum_b) + 1), cum_b, "o-")
ax.axhline(0.80, color="C1", ls="--", label="80%")
ax.axhline(0.85, color="C2", ls="--", label="85%")
ax.set_xlabel("n_components")
ax.set_ylabel("cumulative explained variance")
ax.legend()
ax.set_title("BATTERS: PCA variance vs n_components")
plt.tight_layout()
plt.show()


In [ ]:
PCA_N_COMPONENTS_CANDIDATES_BAT = [8, 10, 12, 14]
K_MIN_BAT = 6
K_MAX_BAT = 12
grid_gmm_bat = grid_sweep_pca_and_gmm(
    X_scaled_bat,
    pca_n_components_list=PCA_N_COMPONENTS_CANDIDATES_BAT,
    k_min=K_MIN_BAT,
    k_max=K_MAX_BAT,
    random_state=RANDOM_STATE,
    n_init=N_INIT,
    covariance_type=GMM_COVARIANCE_TYPE,
)
grid_gmm_bat.sort_values("gmm_bic", ascending=True).head(25)

In [ ]:
pick_b = grid_gmm_bat.sort_values("gmm_bic", ascending=True).iloc[0]
PCA_N_BAT = int(pick_b["pca_n_components"])
K_BAT = int(pick_b["k"])
print("Suggested [batter] (min BIC): PCA_N =", PCA_N_BAT, ", n_components =", K_BAT)

prod_cfg_bat = ArchetypeClusteringConfig(
    pca_n_components=PCA_N_BAT,
    n_clusters=K_BAT,
    random_state=RANDOM_STATE,
    n_init=N_INIT,
    covariance_type=GMM_COVARIANCE_TYPE,
)
prod_cfg_bat

In [ ]:
labeled_bat, meta_bat, _bundle_bat = fit_archetype_clustering(
    gold_df_bat, role=ROLE_BAT, year=YEAR, config=prod_cfg_bat
)
print("[batter]", meta_bat["clustering_method"], "BIC", round(meta_bat["gmm_bic"], 2))
labeled_bat[["player_id", "year", "role", "cluster_id"]].head()

### Combined `ArchetypeClusteringConfigsByRole` (pitcher vs batter)

In [ ]:
pipeline_configs_by_role = ArchetypeClusteringConfigsByRole(
    pitcher=prod_cfg,
    batter=prod_cfg_bat,
)
pipeline_configs_by_role